Downloading the data from the Unraveled tour first, to see what the SeatData API can offer us and what we can do with the provided schema.

### Setup

In [1]:
import os
import time
import glob
import requests
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from google.colab import userdata

drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/olivia_rodrigo_data"
RAW_DIR = os.path.join(OUTPUT_DIR, "daily_by_event")
os.makedirs(RAW_DIR, exist_ok=True)

API_KEY = userdata.get("api_key_seat")
BASE_URL = "https://seatdata.io/api/v1"
HEADERS = {"Authorization": f"Bearer {API_KEY}"}


Mounted at /content/drive


**We start by querying the search endpoint for "Olivia Rodrigo" concerts in October 2026 to figure out what identifiers the API returns.**

In [ ]:
response = requests.get(
    f"{BASE_URL}/events/search",
    headers=HEADERS,
    params={
        "event_name": "Olivia Rodrigo",
        "event_date": "2026-10",
        "historical": "true",
        "limit": 10,
    },
    timeout=30
)

initial_events = response.json().get("data", [])
df_initial = pd.DataFrame(initial_events)

display(df_initial.head())

,event_id,event_name,event_date,event_time,tm_event_id,sh_event_id,vs_event_id,days_on_seatdata,first_seen_date,venue_name,...,venue_country,venue_lat,venue_lng,venue_slug,venue_tm_id,std_event_id,std_venue_id,performer,event_type,tour_name
0,1338351,Olivia Rodrigo,2026-10-03,19:00:00,1kA8v0PdGACx387,161041524,6969246.0,190,2026-04-30,Capital One Arena,...,United States,38.897412,-77.020029,capital-one-arena,KovZpaKuJe,10384,43,Olivia Rodrigo,concert,None
1,1338353,Olivia Rodrigo,2026-10-04,19:00:00,1AvfZ_aGkVqmncZ,161041519,6969248.0,190,2026-04-30,Capital One Arena,...,United States,38.897412,-77.020029,capital-one-arena,KovZpaKuJe,10384,43,Olivia Rodrigo,concert,None
2,1338363,Olivia Rodrigo,2026-10-07,19:00:00,G5eVZ_aUyED-b,161041518,6969250.0,190,2026-04-30,Spectrum Center,...,United States,35.225222,-80.839921,spectrum-center,KovZpZA6AEIA,10384,441,Olivia Rodrigo,concert,None
3,1338355,Olivia Rodrigo,2026-10-08,19:00:00,G5eVZ_aUyt5-g,161041522,6969252.0,190,2026-04-30,Spectrum Center,...,United States,35.225222,-80.839921,spectrum-center,KovZpZA6AEIA,10384,441,Olivia Rodrigo,concert,None
4,1338373,Olivia Rodrigo,2026-10-11,19:00:00,vv178Z_aGkYeTUOa,161041523,6969254.0,190,2026-04-30,United Center,...,United States,41.881244,-87.674274,united-center,KovZpa2M7e,10384,83,Olivia Rodrigo,concert,None


**There seems to be a standard id for all of the events 10384, if this is grouping all the events for the Unraveled Tour then our job will be much more easy. The UI doesn't group them in anyway so I wasn't expecting to be able to locate each date so easily.**

**Using the standardized tour ID (`std_event_id`), we extract all concerts mapped to this tour**

In [ ]:
std_tour_id = 10384
all_events = []
params = {"std_event_id": std_tour_id, "historical": "true", "limit": 200}

while True:
    resp = requests.get(f"{BASE_URL}/events/search", headers=HEADERS, params=params, timeout=30)
    result = resp.json()
    all_events.extend(result.get("data", []))
    cursor = result.get("next_cursor")
    if not cursor:
        break
    params["starting_after"] = cursor

tour_df = pd.DataFrame(all_events)
tour_df["event_date"] = pd.to_datetime(tour_df["event_date"])

#filter to standard tour window boundaries
tour_df = tour_df[tour_df["event_date"].between("2026-09-25", "2027-05-27")].drop_duplicates(subset=["event_id"]).sort_values("event_date").reset_index(drop=True)

print(f"Total unique tour dates discovered in API: {len(tour_df)}")
display(tour_df[['event_id', 'event_date', 'venue_name', 'venue_city']].head(5))

Total unique tour dates discovered in API: 86


,event_id,event_date,venue_name,venue_city
0,1338356,2026-09-25,PeoplesBank Arena (Formerly XL Center),Hartford
1,1338368,2026-09-26,PeoplesBank Arena (Formerly XL Center),Hartford
2,1338348,2026-09-29,PPG Paints Arena,Pittsburgh
3,1338350,2026-09-30,PPG Paints Arena,Pittsburgh
4,1338351,2026-10-03,Capital One Arena,Washington


**Lets see what we can get per event**

In [ ]:
sample_event_id = int(tour_df.iloc[0]["event_id"])
sample_url = f"{BASE_URL}/events/{sample_event_id}/stats"
sample_resp = requests.get(sample_url, headers=HEADERS, params={"limit": 10}, timeout=30)

sample_stats = pd.DataFrame(sample_resp.json().get("data", []))
sample_stats["timestamp"] = pd.to_datetime(sample_stats["timestamp"])

print(f"Sample data retrieved for Event {sample_event_id}:")
display(sample_stats[["timestamp", "avg_price", "median_price", "get_in"]].head(5))


Sample data retrieved for Event 1338356:


,timestamp,avg_price,median_price,get_in
0,2026-09-25 22:04:09+00:00,750.074,246.84,69.13
1,2026-09-25 14:04:33+00:00,1209.630,394.15,138.77
2,2026-09-25 06:04:26+00:00,1243.920,427.48,180.09
3,2026-09-24 22:09:22+00:00,1184.660,449.14,202.60
4,2026-09-24 14:07:06+00:00,1260.350,483.95,199.15


**There's multiple snapshots per day, we will need to average them later in a cleaner dataset.**

**Because API requests to SeatData cost money, we want to run this code just once. We will save all the data into Google Drive per event, then aggregate and clean afterwards.**

In [ ]:
def download_event_snapshots(event_id, event_name=None, event_date=None, venue=None):
    filename = os.path.join(RAW_DIR, f"event_{event_id}_all_snapshots.csv")
    if os.path.exists(filename):
        return pd.read_csv(filename)

    url = f"{BASE_URL}/events/{event_id}/stats"
    params = {"start_date": "2024-10-01", "end_date": "2027-05-27", "limit": 200}
    snapshots = []

    while True:
        response = requests.get(url, headers=HEADERS, params=params, timeout=60)
        response.raise_for_status()
        result = response.json()
        snapshots.extend(result.get("data", []))
        cursor = result.get("next_cursor")
        if not cursor:
            break
        params["starting_after"] = cursor
        time.sleep(0.1)

    if not snapshots:
        return None

    raw = pd.DataFrame(snapshots)
    raw["timestamp"] = pd.to_datetime(raw["timestamp"], utc=True)
    raw = raw.sort_values("timestamp", ascending=False)

    # Preserve metadata fields
    raw["event_id"] = event_id
    raw["event_name"] = event_name
    raw["event_date"] = event_date
    raw["venue"] = venue

    keep_cols = ["event_id", "event_name", "event_date", "venue", "timestamp", "avg_price", "median_price", "get_in", "total_listings_active"]
    raw = raw[[c for c in keep_cols if c in raw.columns]]
    raw.to_csv(filename, index=False)
    return raw

**Download all the data per event, added a custom variable to avoid high costs and debug in a cheaper way.**

In [ ]:
NUM_EVENTS_TO_DOWNLOAD = 1

events_to_process = tour_df.head(NUM_EVENTS_TO_DOWNLOAD)

for idx, event in events_to_process.iterrows():
    e_id = int(event["event_id"])
    download_event_snapshots(
        event_id=e_id,
        event_name=event.get("event_name"),
        event_date=event.get("event_date"),
        venue=event.get("venue_name")
    )
    if (idx + 1) % 10 == 0 or (idx + 1) == len(events_to_process):
        print(f"Progress: {idx+1}/{len(events_to_process)} events processed and saved to Drive.")

Progress: 1/1 events processed and saved to Drive.


**We consolidate it all into one dataset**

In [ ]:
all_csv_files = glob.glob(os.path.join(RAW_DIR, "event_*_all_snapshots.csv"))

individual_frames = []
for f in all_csv_files:
    df_snap = pd.read_csv(f)
    # Convert timestamps and aggregate to keep the daily end-of-day state
    df_snap["timestamp"] = pd.to_datetime(df_snap["timestamp"], utc=True)
    df_snap["date"] = df_snap["timestamp"].dt.date

    # Aggregate: Keep the latest observation per calendar date
    df_daily = df_snap.sort_values("timestamp", ascending=False).drop_duplicates("date", keep="first")
    individual_frames.append(df_daily)

master_df = pd.concat(individual_frames, ignore_index=True)
master_df["date"] = pd.to_datetime(master_df["date"])
master_df = master_df.sort_values(["event_id", "date"]).reset_index(drop=True)

# Export Compiled Master Dataset
master_path = os.path.join(OUTPUT_DIR, "olivia_rodrigo_all_daily_prices.csv")
master_df.to_csv(master_path, index=False)

print(f"Consolidation complete!")
print(f"Total unique events gathered: {master_df['event_id'].nunique()}")
print(f"Total pricing observations: {len(master_df)}")
display(master_df.head(10))

Consolidation complete!
Total unique events gathered: 1
Total pricing observations: 149


,event_id,event_name,event_date,venue,timestamp,avg_price,median_price,get_in,total_listings_active,date
0,1338356,Olivia Rodrigo,2026-09-25,PeoplesBank Arena (Formerly XL Center),2026-04-30 23:00:04+00:00,1013.870,842.490,268.92,329,2026-04-30
1,1338356,Olivia Rodrigo,2026-09-25,PeoplesBank Arena (Formerly XL Center),2026-05-01 23:00:35+00:00,949.213,810.000,256.50,426,2026-05-01
2,1338356,Olivia Rodrigo,2026-09-25,PeoplesBank Arena (Formerly XL Center),2026-05-02 22:56:53+00:00,928.958,786.500,291.00,439,2026-05-02
3,1338356,Olivia Rodrigo,2026-09-25,PeoplesBank Arena (Formerly XL Center),2026-05-03 22:55:05+00:00,909.995,758.000,291.00,438,2026-05-03
4,1338356,Olivia Rodrigo,2026-09-25,PeoplesBank Arena (Formerly XL Center),2026-05-04 22:50:08+00:00,1061.750,957.900,269.99,434,2026-05-04
5,1338356,Olivia Rodrigo,2026-09-25,PeoplesBank Arena (Formerly XL Center),2026-05-05 22:57:02+00:00,1434.740,881.055,247.84,542,2026-05-05
6,1338356,Olivia Rodrigo,2026-09-25,PeoplesBank Arena (Formerly XL Center),2026-05-06 23:07:56+00:00,1521.360,890.010,247.84,552,2026-05-06
7,1338356,Olivia Rodrigo,2026-09-25,PeoplesBank Arena (Formerly XL Center),2026-05-07 23:29:04+00:00,1020.200,640.200,289.54,639,2026-05-07
8,1338356,Olivia Rodrigo,2026-09-25,PeoplesBank Arena (Formerly XL Center),2026-05-08 23:09:19+00:00,991.229,603.825,274.54,662,2026-05-08
9,1338356,Olivia Rodrigo,2026-09-25,PeoplesBank Arena (Formerly XL Center),2026-05-09 23:10:34+00:00,984.953,598.400,266.27,672,2026-05-09
